# Supplementary Figures (S1–S14)

Reproduces SI figures from Rappeport & Nitzan, *Fitness and Overfitness*.

Each section loads pre-computed simulation output from `../data/`. To regenerate, run the matching script in `../experiments/`, e.g.:

```bash
python -m experiments.si_alt_fitness
```

See `../data/README.md` for filename → figure mapping and flagged inconsistencies.

## 0. Setup

In [ ]:
import os, sys, pickle, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath('../src'))
from overfitness_paper import (
    setup_plot_style, plot_bubble_chart, plot_env_change_heatmap,
    plot_invasion_heatmap, plot_muller, plot_complexity_trajectory,
)
DATA = os.path.abspath('../data')
FIG  = os.path.abspath('../figures/supplementary'); os.makedirs(FIG, exist_ok=True)
setup_plot_style()

## Fig S1 — Mean class fitness ⟨p_q⟩ vs class complexity (linear, multi-peak)

**Inconsistency flagged**: SI caption states σ=0.1 but the source code uses σ=1.0. Script keeps σ=1.0; data/README.md notes this for the manuscript.

In [ ]:
d = np.load(f'{DATA}/s1_mean_fitness.npz')
ks, q_stars, means, sems = d['ks'], d['q_stars'], d['means'], d['sems']
fig, ax = plt.subplots(figsize=(6, 4))
for i, q_star in enumerate(q_stars):
    ax.errorbar(ks, means[i], yerr=1.96 * sems[i], label=f'$q^*={q_star}$', marker='o')
ax.set_xlabel('Class complexity $q$'); ax.set_ylabel(r'Mean class fitness $\langle p_q \rangle$')
ax.legend()
fig.savefig(f'{FIG}/figS1.pdf', bbox_inches='tight'); plt.show()

## Fig S2 — Mean class fitness for two-layer NNs

In [ ]:
# Same shape as S1; data lives in s2_mean_fitness_nn.npz (computed by si_neural_network.py).
path = f'{DATA}/s2_mean_fitness_nn.npz'
if os.path.exists(path):
    d = np.load(path)
    fig, ax = plt.subplots(figsize=(6, 4))
    for i, q_star in enumerate(d['q_stars']):
        ax.errorbar(d['ks'], d['means'][i], yerr=1.96 * d['sems'][i],
                    label=f'$q^*={q_star}$', marker='o')
    ax.set_xlabel('Hidden-layer size $q$'); ax.set_ylabel(r'$\langle p_q \rangle$'); ax.legend()
    fig.savefig(f'{FIG}/figS2.pdf', bbox_inches='tight'); plt.show()
else:
    print(f'Missing {path}. Run: python -m experiments.si_neural_network (then add the mean-fitness output)')

## Fig S3 — Polynomials: selected complexity (stable, S3A) and env-change heatmap (S3B)

In [ ]:
d = np.load(f'{DATA}/s3_polynomial.npz')
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_bubble_chart(d['bubble'], ax=axes[0], xlabel='$q^*$', ylabel='Selected $q$')
plot_env_change_heatmap(d['heatmap'], d['ks'], d['n_envs_list'], ax=axes[1])
fig.savefig(f'{FIG}/figS3.pdf', bbox_inches='tight'); plt.show()

## Fig S4 — Neural networks: selected complexity (S4A) and env-change heatmap (S4B)

Uses γ=0.01 (override of the global default; documented in `config.py`).

In [ ]:
d = np.load(f'{DATA}/s4_nn.npz')
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_bubble_chart(d['bubble'], ax=axes[0], xlabel='$q^*$', ylabel='Selected $q$')
axes[0].set_xticklabels(d['ks']); axes[0].set_yticklabels(d['ks'])
im = axes[1].imshow(d['heatmap'], aspect='auto', origin='lower', cmap='inferno')
axes[1].set_xticks(range(len(d['true_ks']))); axes[1].set_xticklabels(d['true_ks'])
axes[1].set_yticks(range(len(d['change_rates']))); axes[1].set_yticklabels(d['change_rates'])
axes[1].set_xlabel('$q^*$'); axes[1].set_ylabel('env change rate')
plt.colorbar(im, ax=axes[1])
fig.savefig(f'{FIG}/figS4.pdf', bbox_inches='tight'); plt.show()

## Fig S5 — AR(1) correlated environments (linear top, NN bottom)

Three columns: static iid / changing iid / changing AR(1) ρ=0.99.

In [ ]:
lin = np.load(f'{DATA}/s5_ar1_linear.npz')
nn  = np.load(f'{DATA}/s5_ar1_nn.npz')
labels = ['static_iid', 'changing_iid', 'changing_ar1_99']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for j, label in enumerate(labels):
    plot_bubble_chart(lin[label], ax=axes[0, j], xlabel='$q^*$', ylabel='Selected $q$')
    axes[0, j].set_title(f'Linear: {label}')
    plot_bubble_chart(nn[label], ax=axes[1, j], xlabel='$q^*$', ylabel='Selected $q$')
    axes[1, j].set_xticklabels(nn['ks']); axes[1, j].set_yticklabels(nn['ks'])
    axes[1, j].set_title(f'NN: {label}')
fig.savefig(f'{FIG}/figS5.pdf', bbox_inches='tight'); plt.show()

## Fig S6 — Correlated cues (linear top, NN bottom)

Three columns: iid cues / AR(1) cues ρ=0.99 / constant cue.

In [ ]:
lin = np.load(f'{DATA}/s6_cues_linear.npz')
nn  = np.load(f'{DATA}/s6_cues_nn.npz')
labels = ['iid', 'ar1_99', 'constant']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for j, label in enumerate(labels):
    plot_bubble_chart(lin[label], ax=axes[0, j], xlabel='$q^*$', ylabel='Selected $q$')
    axes[0, j].set_title(f'Linear cues: {label}')
    plot_bubble_chart(nn[label], ax=axes[1, j], xlabel='$q^*$', ylabel='Selected $q$')
    axes[1, j].set_xticklabels(nn['ks']); axes[1, j].set_yticklabels(nn['ks'])
    axes[1, j].set_title(f'NN cues: {label}')
fig.savefig(f'{FIG}/figS6.pdf', bbox_inches='tight'); plt.show()

## Fig S7 — Effect of class size on selection accuracy

In [ ]:
d = np.load(f'{DATA}/s7_class_size.npz')
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(d['class_sizes'], d['accuracy'], marker='o')
ax.set_xscale('log'); ax.set_xlabel('Class size'); ax.set_ylabel('Selection accuracy')
ax.axhline(1.0 / 9, ls='--', color='gray', label='random')
ax.legend()
fig.savefig(f'{FIG}/figS7.pdf', bbox_inches='tight'); plt.show()

## Fig S8 — Invasion experiments

In [ ]:
with open(f'{DATA}/s8_mullers.pkl', 'rb') as f:
    mullers = pickle.load(f)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
# Row 0: example invasion Mullers
for i, label in enumerate(['q3', 'q7']):
    log = mullers[label]
    plot_muller(log['class_frequency'].T, ax=axes[0, i])
    axes[0, i].set_title(label); axes[0, i].set_xlabel('Generations')
# Row 1: invasion success heatmaps for two change rates
for i, label in enumerate(['change0', 'change02']):
    success = np.load(f'{DATA}/s8_invasion_{label}.npy')
    plot_invasion_heatmap(success, ks=np.arange(1, success.shape[0] + 1), ax=axes[1, i],
                          title=f'env_change_rate={0 if label=="change0" else 0.2}')
fig.savefig(f'{FIG}/figS8.pdf', bbox_inches='tight'); plt.show()

## Fig S9 — Mutation-driven complexity evolution

Panels: trajectories from q₀=1 and q₀=9; 3×3 (μ_b, μ_w) grid at q*=5.

In [ ]:
q1 = np.load(f'{DATA}/s9_q1.npz')
q9 = np.load(f'{DATA}/s9_q9.npz')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, d, title in [(axes[0], q1, 'q₀=1'), (axes[1], q9, 'q₀=9')]:
    for key in d.files:
        traj = d[key]  # (n_real, T+1)
        mean = traj.mean(0)
        sd = traj.std(0)
        ax.plot(mean, label=key)
        ax.fill_between(np.arange(len(mean)), mean - sd, mean + sd, alpha=0.2)
    ax.set_title(title); ax.set_xlabel('Generations'); ax.set_ylabel('Avg complexity')
    ax.legend(fontsize=7)
fig.savefig(f'{FIG}/figS9_trajectories.pdf', bbox_inches='tight'); plt.show()

In [ ]:
grid = np.load(f'{DATA}/s9_grid.npz')
mus_b = sorted(set(float(k.split('_')[0][2:]) for k in grid.files))
mus_w = sorted(set(float(k.split('_')[1][2:]) for k in grid.files))
fig, axes = plt.subplots(len(mus_b), len(mus_w), figsize=(12, 10), sharex=True, sharey=True)
for i, mb in enumerate(mus_b):
    for j, mw in enumerate(mus_w):
        key = f'mb{mb}_mw{mw}'
        if key not in grid.files:
            continue
        traj = grid[key]
        m, s = traj.mean(0), traj.std(0)
        axes[i, j].plot(m); axes[i, j].fill_between(np.arange(len(m)), m - s, m + s, alpha=0.2)
        axes[i, j].set_title(f'μ_b={mb}, μ_w={mw}', fontsize=8)
fig.savefig(f'{FIG}/figS9_grid.pdf', bbox_inches='tight'); plt.show()

## Fig S10 — Selected complexity in stable envs, alternative fitness functions

In [ ]:
d = np.load(f'{DATA}/s10_alt_fitness.npz')
fts = ['L2', 'L1', 'rational', 'cutoff']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, ft in zip(axes, fts):
    plot_bubble_chart(d[ft], ax=ax, xlabel='$q^*$', ylabel='Selected $q$')
    diag = np.mean(np.diag(d[ft]))
    ax.set_title(f'{ft} (acc={diag:.1%})')
fig.savefig(f'{FIG}/figS10.pdf', bbox_inches='tight'); plt.show()

## Fig S11 — Heatmap over (q*, env change rate), alternative fitness functions

In [ ]:
d = np.load(f'{DATA}/s11_alt_fitness_change.npz')
fts = ['L2', 'L1', 'rational', 'cutoff']
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, ft in zip(axes, fts):
    plot_env_change_heatmap(d[ft], d['ks'], d['n_envs_list'], ax=ax)
    ax.set_title(ft)
fig.savefig(f'{FIG}/figS11.pdf', bbox_inches='tight'); plt.show()

## Fig S12 — Selected complexity in stable envs, alternative noise distributions

In [ ]:
d = np.load(f'{DATA}/s12_alt_noise.npz')
nts = ['gaussian', 'uniform', 'laplacian']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, nt in zip(axes, nts):
    plot_bubble_chart(d[nt], ax=ax, xlabel='$q^*$', ylabel='Selected $q$')
    diag = np.mean(np.diag(d[nt]))
    ax.set_title(f'{nt} (acc={diag:.1%})')
fig.savefig(f'{FIG}/figS12.pdf', bbox_inches='tight'); plt.show()

## Fig S13 — Heatmap over (q*, env change rate), alternative noise distributions

In [ ]:
d = np.load(f'{DATA}/s13_alt_noise_change.npz')
nts = ['gaussian', 'uniform', 'laplacian']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, nt in zip(axes, nts):
    plot_env_change_heatmap(d[nt], d['ks'], d['n_envs_list'], ax=ax)
    ax.set_title(nt)
fig.savefig(f'{FIG}/figS13.pdf', bbox_inches='tight'); plt.show()

## Fig S14 — Parameter sensitivity: γ, σ, class size, T

Red marks indicate the value used in the main results.

In [ ]:
d = np.load(f'{DATA}/s14_sensitivity.npz')
from overfitness_paper.config import FITNESS_GAMMA, SIGMA, CLASS_SIZE as CS, T as TT
panels = [
    ('gamma', FITNESS_GAMMA, 'Selection strength γ'),
    ('sigma', SIGMA,         'Noise level σ'),
    ('class_size', CS,       'Population per class'),
    ('T', TT,                'Generations'),
]
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, (key, mark, label) in zip(axes.flat, panels):
    xs = d[key + '_values']; ys = d[key + '_accuracy']
    ax.errorbar(xs, ys, marker='o')
    ax.axvline(mark, color='red', label='main results')
    ax.axhline(1.0 / 9, ls=':', color='gray', label='random')
    ax.set_xscale('log'); ax.set_xlabel(label); ax.set_ylabel('Selection accuracy')
    ax.legend(fontsize=7)
fig.savefig(f'{FIG}/figS14.pdf', bbox_inches='tight'); plt.show()